In [10]:
import os
import sys
sys.path.append('..')
import torch
import librosa
from scipy.io import wavfile
from IPython import display as disp
from pathlib import Path
from tqdm import tqdm

from src.modules import ResynthesisModule, SpeakerModule
from anonymization.anonymizer import DemoAnonymizer

/mnt/nvme-data1/waris/PSI-TAMU/DarkStream/notebooks/../anonymization/demo_speaker_embeddings.py:5: UserWarning: Module 'speechbrain.pretrained' was deprecated, redirecting to 'speechbrain.inference'. Please update your script. This is a change from SpeechBrain 1.0. See: https://github.com/speechbrain/speechbrain/releases/tag/v1.0.0
  from speechbrain.pretrained import EncoderClassifier


In [2]:

os.environ["CUDA_VISIBLE_DEVICES"] = "3" # Set the GPU device
MAX_WAV_VALUE = 32768.0

### Note
Use resynthesis_nola for 0ms lookahead, resynthesis_la7 for 140ms lookahead, and resynthesis_la14 for 280ms lookahead model

In [19]:
speaker_encoder = SpeakerModule(device="cuda")
resynthesizer = ResynthesisModule.load_from_checkpoint(
    "../ckpts/resynthesis_la14.ckpt",
    encoder_path=None,
    map_location="cuda"
)
resynthesizer.eval()

INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch custom.py: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch classifier.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cached
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder
INFO:speechbrain.utils.fetching:Fetch hyperpara

ResynthesisModule(
  (encoder): PasrModule(
    (net): HiFiEncoder(
      (conv_pre): CausalConv(
        (layer): ParametrizedRawStreamingConv1d(
          1, 16, kernel_size=(7,), stride=(1,)
          (parametrizations): ModuleDict(
            (weight): ParametrizationList(
              (0): _WeightNorm()
            )
          )
        )
      )
      (downs): ModuleList(
        (0): CausalConv(
          (layer): ParametrizedRawStreamingConv1d(
            16, 32, kernel_size=(4,), stride=(2,)
            (parametrizations): ModuleDict(
              (weight): ParametrizationList(
                (0): _WeightNorm()
              )
            )
          )
        )
        (1): CausalConv(
          (layer): ParametrizedRawStreamingConv1d(
            32, 64, kernel_size=(4,), stride=(2,)
            (parametrizations): ModuleDict(
              (weight): ParametrizationList(
                (0): _WeightNorm()
              )
            )
          )
        )
        (2): 

In [20]:
wav_file = "../sample_wav/BDL/arctic_a0001.wav"
_device = "cuda"
sample_rate = 16000

audio, _ = librosa.load(wav_file, sr=sample_rate)

disp.Audio(audio, rate=16000)

### Non streaming synthesis

In [21]:
audio = torch.from_numpy(audio).unsqueeze(0).unsqueeze(0).to(_device)
embed = speaker_encoder(audio.squeeze(0))
resyth_audio = resynthesizer(audio, embed)
resyth_audio = resyth_audio.squeeze()
resyth_audio = resyth_audio * MAX_WAV_VALUE
resyth_audio = resyth_audio.detach().cpu().numpy().astype('int16')

disp.Audio(resyth_audio, rate=16000)

### Streaming synthesis (One way)

In [22]:
resynthesizer.eval()
with torch.no_grad():
    original_length = audio.shape[-1]
    # Pad audio for lookahead
    # frame_size = int(resynthesizer_mm.hparams.sampling_rate // resynthesizer_mm.hparams.frame_rate)
    frame_size = 320

    # frame_size = int(encoder_mm.net.sample_rate // encoder_mm.net.frame_rate)
    all_audios = []
    all_z_embeds = []

    with resynthesizer.encoder.net.streaming(batch_size=1):
        with resynthesizer.decoder.streaming(batch_size=1):
            for offset in tqdm(range(0, original_length, frame_size)):
                frame = audio[:, :, offset:offset + frame_size]
                _, z_embed = resynthesizer.encoder.net(frame)
                if z_embed.shape[-1] == 0:
                    continue
                resynth_audio_chunk = resynthesizer.decoder.decode(z_embed, embed)
                all_audios.append(resynth_audio_chunk)   
                all_z_embeds.append(z_embed)    
    
all_audios = torch.cat(all_audios, dim=-1)
resyth_audio = all_audios.squeeze().cpu().numpy()
resyth_audio = resyth_audio * MAX_WAV_VALUE
resyth_audio = resyth_audio.astype('int16')

disp.Audio(resyth_audio, rate=16000)

  6%|▌         | 10/162 [00:00<00:03, 47.14it/s]

100%|██████████| 162/162 [00:07<00:00, 22.55it/s]


### Streaming synthesis (Another way)

In [7]:
resynthesizer.encoder.net.streaming_forever(batch_size=1)
resynthesizer.decoder.streaming_forever(batch_size=1)
with torch.no_grad():
    original_length = audio.shape[-1]
    # Pad audio for lookahead
    # frame_size = int(resynthesizer_mm.hparams.sampling_rate // resynthesizer_mm.hparams.frame_rate)
    frame_size = 320

    # frame_size = int(encoder_mm.net.sample_rate // encoder_mm.net.frame_rate)
    all_audios = []
    all_z_embeds = []

    for offset in tqdm(range(0, original_length, frame_size)):
        frame = audio[:, :, offset:offset + frame_size]
        _, z_embed = resynthesizer.encoder.net(frame)
        if z_embed.shape[-1] == 0:
            continue
        resynth_audio_chunk = resynthesizer.decoder.decode(z_embed, embed)
        all_audios.append(resynth_audio_chunk)   
        all_z_embeds.append(z_embed)    
    
all_audios = torch.cat(all_audios, dim=-1)
resyth_audio = all_audios.squeeze().cpu().numpy()
resyth_audio = resyth_audio * MAX_WAV_VALUE
resyth_audio = resyth_audio.astype('int16')

disp.Audio(resyth_audio, rate=16000)

100%|██████████| 162/162 [00:08<00:00, 19.80it/s]


### Anonymization
For anonymization use embedding from anonymizer model instead

In [11]:
anonymizer = DemoAnonymizer(
    model_path=Path('../anonymization/models'), 
    model_tag='gan',
    device="cuda"
)

INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch custom.py: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


Model type of anonymizer: gan


INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch classifier.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder
INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch custom.py: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch embedding

In [16]:
anon_embed = anonymizer.anonymize_embedding(audio.squeeze().cpu().numpy(), 16000)
anon_embed = torch.cat([anon_embed[192:], anon_embed[:192]], dim=0) #embeds are concatenated in the reversed order

In [18]:
resynthesizer.eval()
with torch.no_grad():
    original_length = audio.shape[-1]
    # Pad audio for lookahead
    # frame_size = int(resynthesizer_mm.hparams.sampling_rate // resynthesizer_mm.hparams.frame_rate)
    frame_size = 320

    # frame_size = int(encoder_mm.net.sample_rate // encoder_mm.net.frame_rate)
    all_audios = []
    all_z_embeds = []

    with resynthesizer.encoder.net.streaming(batch_size=1):
        with resynthesizer.decoder.streaming(batch_size=1):
            for offset in tqdm(range(0, original_length, frame_size)):
                frame = audio[:, :, offset:offset + frame_size]
                _, z_embed = resynthesizer.encoder.net(frame)
                if z_embed.shape[-1] == 0:
                    continue
                resynth_audio_chunk = resynthesizer.decoder.decode(z_embed, anon_embed.unsqueeze(0))
                all_audios.append(resynth_audio_chunk)   
                all_z_embeds.append(z_embed)    
    
all_audios = torch.cat(all_audios, dim=-1)
resyth_audio = all_audios.squeeze().cpu().numpy()
resyth_audio = resyth_audio * MAX_WAV_VALUE
resyth_audio = resyth_audio.astype('int16')

disp.Audio(resyth_audio, rate=16000)

100%|██████████| 162/162 [00:07<00:00, 20.77it/s]
